In [2]:
import torch
from torch import nn
from torch.nn import functional as F
import math

CNN(DoubleConv/Down/Up)

In [3]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

In [4]:
class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = DoubleConv(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        before_pool = self.conv(x)
        after_pool = self.pool(before_pool)

        return before_pool, after_pool

In [5]:
class Up(nn.Module):
    def __init__(self, in_channels, out_channels, encoder_channels=None):
        super().__init__()
        if encoder_channels is None:
            encoder_channels = out_channels
        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.conv = DoubleConv(encoder_channels + out_channels, out_channels)

    def forward(self, x_encoder, x_decoder):
        x_up = self.up(x_decoder)

        diff_h = x_encoder.size(2) - x_up.size(2)
        diff_w = x_encoder.size(3) - x_up.size(3)

        x_up = F.pad(x_up, [
            diff_w // 2, diff_w - diff_w // 2,
            diff_h // 2, diff_h - diff_h // 2
        ])

        x = torch.cat([x_encoder, x_up], dim=1)

        x = self.conv(x)

        return x

Transformer(SelfAttention/MultiHeadAttention/FeedForward)

In [6]:
class SelfAttention(nn.Module):
    def __init__(self, dropout=0.1):
        super().__init__()
        self.softmax = nn.Softmax(dim=-1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, Q, K, V):
        d_k = Q.size(-1)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

        attn = self.softmax(scores)

        attn = self.dropout(attn)

        out = torch.matmul(attn, V)

        return out

In [7]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, \
            f"d_model should be divisible by n_heads"
        self.d_k = d_model // n_heads
        self.n_heads = n_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        self.self_attention = SelfAttention(dropout)
        self.fc = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v):
        batch_size = q.size(0)

        Q = self.W_q(q).view(batch_size, -1, self.n_heads, self.d_k).view(1, 2)
        K = self.W_k(k).view(batch_size, -1, self.n_heads, self.d_k).view(1, 2)
        V = self.W_v(v).view(batch_size, -1, self.n_heads, self.d_k).view(1, 2)

        out = self.self_attention(Q, K, V)

        out = out.transpose(1, 2).contiguous().view(batch_size, -1, self.n_heads, self.d_k)

        out = self.fc(out)

        out = self.dropout(out)

        return out

In [8]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.fc2(F.gelu(self.fc1(x))))

In [9]:
class TransformerEncoder(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout)

    def forward(self, src):
        out = self.attn(self.norm1(src), self.norm1(src), self.norm1(src))
        out = self.ffn(self.norm2(out))

        return out

PatchEmbed

In [10]:
class PatchEmbed(nn.Module):
    def __init__(self, img_size, patch_size, in_channels, d_model):
        super().__init__()
        assert img_size % patch_size == 0, \
            f"img_size should be divisible by patch_size"
        self.n_patches = (img_size // patch_size) **2
        self.projection = nn.Conv2d(in_channels, d_model, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.projection(x)
        x = x.flatten(2)

        return x.transpose(1, 2)

TransUNet模型

In [ ]:
class TransUNet(nn.Module):
    def __init__(self, in_channels=3, n_classes=1, cnn_features=[64, 128, 256], patch_size=16, img_size=224, d_model=512, n_heads=8, d_ff=2048, num_layers=12, dropout=0.1):
        super().__init__()
        self.enc1 = Down(in_channels, cnn_features[0])
        self.enc2 = Down(cnn_features[0], cnn_features[1])
        self.enc3 = Down(cnn_features[1], cnn_features[2])

        img_size_after_cnn = img_size // 8
        self.patch_embed = PatchEmbed(
            img_size_after_cnn,
            patch_size,
            in_channels,
            d_model
        )

        n_patches = self.patch_embed.n_patches

        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches, d_model))

        self.transformer_layers = nn.ModuleList([
            TransformerEncoder(d_model, n_heads, d_ff, dropout) for _ in range(num_layers)
        ])

        self.decoder_projection = nn.Sequential(
            nn.Conv2d(d_model, cnn_features[2], kernel_size=3, padding=1),
            nn.BatchNorm2d(cnn_features[2]),
            nn.ReLU(inplace=True)
        )

        self.dec3 = Up(cnn_features[2], cnn_features[1], encoder_channels=cnn_features[2])
        self.dec2 = Up(cnn_features[1], cnn_features[0], encoder_channels=cnn_features[1])
        self.dec1 = Up(cnn_features[0], cnn_features[0], encoder_channels=cnn_features[0])

        self.head = nn.Conv2d(cnn_features[0], n_classes, kernel_size=1)

        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.apply(self._init_weight)

    def _init_weight(self, x):
        if isinstance(x, nn.Linear):
            nn.init.trunc_normal_(x.weight, std=0.02)
            if x.bias is not None:
                nn.init.constant_(x.bias, 0)
        elif isinstance(x, nn.LayerNorm):
            nn.init.constant_(x.weight, 1.0)
            nn.init.constant_(x.bias, 0)

    def forward(self, x):
        B = x.size(0)
        H, W = x.size(2), x.size(3)

        enc1, x = self.enc1(x)
        enc2, x = self.enc2(x)
        enc3, x = self.enc3(x)

        x = self.patch_embed(x)

        x = x + self.pos_embed

        for layer in self.transformer_layers:
            x = layer(x)

        h = H // 8
        w = W // 8

        x = x.transpose(1, 2)

        x = x.view(B, -1, h, w)

        x = self.decoder_projection(x)

        x = self.dec3(enc3, x)
        x = self.dec2(enc2, x)
        x = self.dec1(enc1, x)

        return self.head(x)